# Predicting Electric Vehicle Purchases — Playground Series S6E9

**Goal:** predict `Will_Buy_EV` (Yes/No) for each customer, submitted as a probability and scored on **ROC-AUC**.

**Approach in this notebook**

1. Understand the data: shapes, dtypes, missingness, target balance, train-vs-test drift.
2. Engineer features in stages — interaction/ratio features, digit-decomposition features, and frequency
   encoding — and **prove each stage earns its place** against a noise-column baseline, logged in a single
   ablation table at the end of Section 3.
3. Add fold-safe target encoding for the categorical columns, shared across all models.
4. Train four diverse models (LightGBM, XGBoost, CatBoost, Logistic Regression) on the **same** 5-fold
   stratified split, using out-of-fold (OOF) predictions for honest validation, with LightGBM's
   hyperparameters chosen by a short Optuna search.
5. Combine them via whichever wins on OOF AUC: an OOF-optimized weighted blend, or a stacking meta-model.

**Why these specific techniques:** the digit-decomposition and frequency-encoding steps, and the general shape
of the modeling pipeline, are validated here against this exact competition — two independent public solutions
(one bronze, one silver medal) each found real OOF gains from them, discovered independently of each other.
Rather than copy either solution's code, this notebook reimplements the same ideas from scratch with its own
noise-bar gating, so you can see exactly which parts of the gain are real on *your* run rather than assumed
from someone else's leaderboard screenshot.


In [ ]:
import os, glob, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_FOLDS = 5
TARGET = "Will_Buy_EV"
ID_COL = "id"

# None of LightGBM/XGBoost/CatBoost use the GPU by default -- you have to ask for it explicitly.
# Set this True on a GPU session (P100/T4/etc); it falls back to CPU automatically if no GPU is found.
USE_GPU = True
try:
    import subprocess
    _has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    _has_gpu = False
USE_GPU = USE_GPU and _has_gpu
print(f"GPU available: {_has_gpu}  ->  USE_GPU = {USE_GPU}")

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()

PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#56B4E9", "#CC79A7"]
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})

pd.set_option("display.max_columns", 50)


## 1. Load the data

In [ ]:
def find_data_dir(slug="playground-series-s6e9"):
    candidates = [f"/kaggle/input/{slug}", f"/kaggle/input/competitions/{slug}", "data", "."]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train.csv")):
            return c
    hits = glob.glob(f"/kaggle/input/**/train.csv", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError("Could not locate train.csv — attach the competition data or set the path by hand.")

DATA_DIR = find_data_dir()
print("Using data from:", DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub   = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("train:", train.shape, " test:", test.shape, " sample_submission:", sub.shape)
sub.head(3)


In [ ]:
FEATURES = [c for c in test.columns if c != ID_COL]
# Use is_numeric_dtype rather than checking for dtype == "object": pandas versions differ on
# whether a string column reads back as "object" or a dedicated string dtype, and this way the
# split is correct either way.
CAT_FEATS = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(train[c])]
NUM_FEATS = [c for c in FEATURES if c not in CAT_FEATS]
print(f"{len(FEATURES)} features -> {len(NUM_FEATS)} numeric, {len(CAT_FEATS)} categorical")
print("numeric:", NUM_FEATS)
print("categorical:", CAT_FEATS)


### At a glance

In [ ]:
_kpis = [
    ("Training rows", f"{len(train):,}"),
    ("Test rows", f"{len(test):,}"),
    ("Features", f"{len(FEATURES)}  ({len(NUM_FEATS)} numeric · {len(CAT_FEATS)} categorical)"),
    ("Positive rate", f"{train[TARGET].eq('Yes').mean():.1%}  (target: {TARGET})"),
]
_cards = "".join(
    f'<div style="background:#F7FAFC;border:1px solid #D9E2EC;border-radius:12px;'
    f'padding:14px 16px;min-width:160px">'
    f'<div style="font-size:11px;letter-spacing:.08em;text-transform:uppercase;'
    f'color:#627D98;font-weight:700">{label}</div>'
    f'<div style="font-size:20px;color:#102A43;font-weight:800;margin-top:4px">{value}</div></div>'
    for label, value in _kpis
)
from IPython.display import HTML, display
display(HTML(f'<div style="display:flex;gap:12px;flex-wrap:wrap;margin:8px 0 4px">{_cards}</div>'))


## 2. A first look: dtypes, missingness, and every column at a glance

In [ ]:
train.info()


In [ ]:
summary_rows = []
for c in FEATURES:
    vc = train[c].value_counts(normalize=True)
    summary_rows.append((
        c,
        "categorical" if c in CAT_FEATS else "numeric",
        train[c].nunique(),
        f"{train[c].isna().mean()*100:.2f}%",
        f"{test[c].isna().mean()*100:.2f}%" if c in test.columns else "-",
        str(vc.index[0])[:16],
        f"{vc.iloc[0]*100:.1f}%",
    ))
summary = pd.DataFrame(summary_rows, columns=[
    "column", "type", "distinct", "train missing", "test missing", "most common", "its share"
])
summary


In [ ]:
# Low-cardinality numeric columns behave more like ordered categories than continuous measurements.
# Worth knowing before we decide how a model should treat them.
LOWCARD_NUM = [c for c in NUM_FEATS if train[c].nunique() <= 20]
print("Low-cardinality numeric columns (<=20 distinct values):", LOWCARD_NUM)
for c in LOWCARD_NUM:
    print(f"  {c}: {sorted(train[c].unique())}")


### Target distribution

In [ ]:
vc = train[TARGET].value_counts()
shares = vc / vc.sum()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
wedges, _ = ax.pie(
    vc.values, colors=[PALETTE[0], PALETTE[3]][:len(vc)], startangle=90,
    wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2),
)
for w, label, share, count in zip(wedges, vc.index, shares.values, vc.values):
    ang = (w.theta2 + w.theta1) / 2
    x, y = 0.78 * np.cos(np.radians(ang)), 0.78 * np.sin(np.radians(ang))
    ax.annotate(f"{label}\n{share*100:.1f}%  (n={count:,})", (x, y), ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
ax.set_title(f"{TARGET}\nminority class = {shares.min()*100:.1f}% of rows", fontsize=13, pad=14)
ax.set(aspect="equal")
plt.tight_layout(); plt.show()

print(f"Class ratio (majority:minority) = {vc.max()/vc.min():.2f} : 1")
print("This is a moderately imbalanced binary target. Since the metric is ROC-AUC (a ranking metric,")
print("not accuracy), we do NOT need to rebalance classes or pick a threshold — we just need well-")
print("calibrated *relative* ordering of predicted probabilities. class_weight/scale_pos_weight can still")
print("help gradient boosters converge a little faster, so we'll try it, but it is not mandatory here.")


### Numeric features: train vs. test

In [ ]:
# Detect artificial floors first, so the plot below can flag them directly rather than making
# you cross-reference a printed list against a grid of nine little histograms.
FLOORS = {}
for c in NUM_FEATS:
    lo_share = (train[c] == train[c].min()).mean()
    if lo_share > 0.05 and train[c].nunique() > 20:
        FLOORS[c] = (train[c].min(), lo_share)

ncols = 3
nrows = int(np.ceil(len(NUM_FEATS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.1 * nrows))
for ax, c in zip(axes.ravel(), NUM_FEATS):
    lo, hi = train[c].quantile([0.005, 0.995])
    ax.hist(train[c].clip(lo, hi), bins=40, density=True, alpha=0.55, color=PALETTE[0], label="train")
    ax.hist(test[c].clip(lo, hi), bins=40, density=True, alpha=0.55, color=PALETTE[1], label="test")
    if c in FLOORS:
        floor_val, share = FLOORS[c]
        ax.axvline(floor_val, color=PALETTE[3], ls="--", lw=1.4)
        ax.text(0.03, 0.92, f"clipped floor\n{share*100:.0f}% of rows", transform=ax.transAxes,
                fontsize=8, color=PALETTE[3], va="top")
    ax.set_title(c, fontsize=10)
    ax.set_yticks([])
for ax in axes.ravel()[len(NUM_FEATS):]:
    ax.axis("off")
axes.ravel()[0].legend(fontsize=8, frameon=False)
fig.suptitle("Numeric features: train vs. test, with clipped floors flagged", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

if FLOORS:
    for c, (val, share) in FLOORS.items():
        test_share = (test[c] == val).mean()
        print(f"'{c}': {share*100:.1f}% of train sits exactly at {val} "
              f"(test: {test_share*100:.1f}%) — a generation artifact, not a real spike.")
else:
    print("No obvious clipped floors detected.")


### Categorical features vs. the target

In [ ]:
overall_rate = train[TARGET].eq("Yes").mean()
ncols = 3
nrows = int(np.ceil(len(CAT_FEATS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.4 * nrows))
for ax, c in zip(axes.ravel(), CAT_FEATS):
    g = train.groupby(c)[TARGET].agg(rate=lambda s: (s == "Yes").mean(), n="size").sort_values("rate", ascending=False)
    colors = [PALETTE[2] if r >= overall_rate else PALETTE[4] for r in g["rate"]]
    bars = ax.bar(g.index.astype(str), g["rate"], color=colors)
    ax.axhline(overall_rate, color="black", lw=1, ls="--", label=f"overall rate ({overall_rate:.1%})")
    for bar, n in zip(bars, g["n"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"n={n:,}",
                ha="center", va="bottom", fontsize=7.5, color="0.35")
    ax.set_title(c, fontsize=10)
    ax.set_ylabel("P(Will_Buy_EV = Yes)")
    ax.set_ylim(0, max(g["rate"].max() * 1.25, overall_rate * 1.25))
    ax.tick_params(axis="x", rotation=15)
for ax in axes.ravel()[len(CAT_FEATS):]:
    ax.axis("off")
axes.ravel()[0].legend(fontsize=8, frameon=False, loc="upper right")
fig.suptitle("Purchase rate by category — orange bars beat the overall average", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()


### Correlation among numeric features

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(train[NUM_FEATS].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numeric feature correlation")
plt.tight_layout(); plt.show()


### Train vs. test drift, quantified

The train-vs-test histograms in Section 2 give a visual sense of drift, but "looks similar" isn't a number.
Here every column gets one: **ECDF distance** (max gap between the two cumulative distributions) for numeric
columns, **total variation distance** for categorical ones. Both range 0 (identical distributions) to 1
(completely disjoint). A well-behaved train/test split from the same generator should show small values
across the board — anything that stands out is worth a second look before trusting CV to generalize to the
leaderboard.

In [ ]:
def ecdf_distance(a, b):
    grid = np.unique(np.r_[a, b])
    a_sorted, b_sorted = np.sort(a), np.sort(b)
    cdf_a = np.searchsorted(a_sorted, grid, side="right") / len(a)
    cdf_b = np.searchsorted(b_sorted, grid, side="right") / len(b)
    return float(np.max(np.abs(cdf_a - cdf_b)))

def tv_distance(a, b):
    pa = pd.Series(a).value_counts(normalize=True)
    pb = pd.Series(b).value_counts(normalize=True)
    return float(0.5 * pa.sub(pb, fill_value=0).abs().sum())

drift_rows = []
for c in NUM_FEATS:
    d = ecdf_distance(train[c].dropna().values, test[c].dropna().values)
    drift_rows.append((c, d, "numeric (ECDF)"))
for c in CAT_FEATS:
    d = tv_distance(train[c].astype(str).values, test[c].astype(str).values)
    drift_rows.append((c, d, "categorical (TV)"))

drift_df = pd.DataFrame(drift_rows, columns=["feature", "distance", "method"]).sort_values("distance")

fig, ax = plt.subplots(figsize=(8, 5.5))
colors = [PALETTE[0] if m.startswith("numeric") else PALETTE[3] for m in drift_df["method"]]
bars = ax.barh(drift_df["feature"], drift_df["distance"], color=colors)
for bar, v in zip(bars, drift_df["distance"]):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f"  {v:.4f}", va="center", fontsize=8)
ax.set_xlabel("distance (0 = identical distributions)")
ax.set_title("Train vs. test drift per column")
plt.tight_layout(); plt.show()

worst = drift_df.iloc[-1]
print(f"Largest drift: '{worst['feature']}' at {worst['distance']:.4f} ({worst['method']})")
print("For reference, a same-distribution split of train against itself typically lands well under 0.01 —")
print("use that as your own rule of thumb for what counts as 'small' on this run.")


### Single-column signal (mutual information)

This ranks each column by how much it alone tells us about the target. It's a reading order, not a
shortlist: a column can score low here and still matter a lot in combination with another (mutual
information never looks at pairs), and this exact limitation is why we don't stop at this chart —
the gradient boosters below can pick up interactions on their own.


In [ ]:
mi_df = train[FEATURES].copy()
for c in CAT_FEATS:
    mi_df[c] = mi_df[c].astype("category").cat.codes
mi = pd.Series(
    mutual_info_classif(mi_df, (train[TARGET] == "Yes").astype(int),
                        discrete_features=[c in CAT_FEATS for c in FEATURES], random_state=SEED),
    index=FEATURES
).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(mi.index, mi.values, color=[PALETTE[3] if f in CAT_FEATS else PALETTE[0] for f in mi.index])
ax.set_xlabel("mutual information with the target")
ax.set_title("Which columns carry signal on their own?")
plt.tight_layout(); plt.show()

print("Top 3:", list(mi.sort_values(ascending=False).index[:3]))


## 3. Feature engineering — and proving it's real

Before trusting any engineered feature, we need a sense of scale: **how much does the OOF AUC move for a
column that is pure noise?** If an engineered feature's gain is smaller than that, it's not evidence of
anything, it's a re-roll of the same folds. So the plan is:

1. Fit a quick baseline LightGBM on raw features only, 5-fold CV → `baseline_auc`.
2. Add a column of random numbers, refit → tells us the size of a *meaningless* change (`noise_bar`).
3. Add our engineered features, refit → compare the gain against `noise_bar`, not against zero.

All engineered features below are **row-wise** (computed independently per row, with no aggregation over the
training set), so there's no risk of leaking information between train and test or between CV folds.


In [ ]:
def engineer_features(df):
    out = df.copy()

    # Total charging access, and a flag for having none at all nearby.
    out["Total_Charging_Stations"] = out["Charging_Stations_Near_Home"] + out["Charging_Stations_Near_Work"]
    out["No_Charging_Access"] = (out["Total_Charging_Stations"] == 0).astype(int)

    # Income relative to age and commute burden relative to cars owned — simple ratios a tree
    # can't reconstruct from the raw columns via a single split.
    out["Income_per_Age"] = out["Annual_Income_USD"] / out["Age"].clip(lower=1)
    out["Commute_per_Car"] = out["Daily_Commute_km"] / (out["Number_of_Cars_Owned"] + 1)

    # Environmental concern interacting with charging access: a concerned buyer with nowhere to
    # charge is a different story than a concerned buyer with a charger at both ends of the commute.
    out["Concern_x_Charging"] = out["Environmental_Concern_Level"] * (out["Total_Charging_Stations"] + 1)

    # A few more interactions/ratios around concern, subsidy, and income -- these specific crosses
    # come from EDA on this exact dataset (a public bronze-medal solution measured real OOF gains
    # from them independently), reimplemented here in our own encoding.
    out["Subsidy_Flag"] = (out["Subsidy_Available"] == "Yes").astype(int)
    out["Home_Charging_Flag"] = (out["Home_Charging_Possible"] == "Yes").astype(int)
    out["Concern_x_Subsidy"] = out["Environmental_Concern_Level"] * out["Subsidy_Flag"]
    out["Concern_x_Income"] = out["Environmental_Concern_Level"] * out["Annual_Income_USD"]
    out["Subsidy_x_Income"] = out["Subsidy_Flag"] * out["Annual_Income_USD"]
    out["HomeCharging_x_Subsidy"] = out["Home_Charging_Flag"] * out["Subsidy_Flag"]
    out["Income_per_Concern"] = out["Annual_Income_USD"] / (out["Environmental_Concern_Level"] + 1)
    out["Commute_per_Concern"] = out["Daily_Commute_km"] / (out["Environmental_Concern_Level"] + 1)

    # Range_Anxiety_Level reads naturally as ordered (Low < Medium < High); give the model an
    # explicit ordinal version alongside the raw category, in case the ordering itself is useful
    # information a category split can't express directly.
    anxiety_map = {"Low": 0, "Medium": 1, "High": 2}
    if "Range_Anxiety_Level" in out.columns:
        out["Range_Anxiety_Ordinal"] = out["Range_Anxiety_Level"].map(anxiety_map)
        out["Anxiety_x_Subsidy"] = out["Range_Anxiety_Ordinal"] * out["Subsidy_Flag"]

    return out

train_fe = engineer_features(train)
test_fe = engineer_features(test)

ENGINEERED = [c for c in train_fe.columns if c not in train.columns]
print("Engineered features:", ENGINEERED)


In [ ]:
def quick_cv_auc(feature_df, feature_cols, cat_cols, n_folds=N_FOLDS, seed=SEED, n_rounds=300):
    # A fast LightGBM OOF check used only to gate feature ideas against a noise bar
    # (not the final model -- see section 4 for the tuned, fully-fit models).
    y = (train[TARGET] == "Yes").astype(int).values
    X = feature_df[feature_cols].copy()
    for c in cat_cols:
        X[c] = X[c].astype("category")

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    for tr_idx, va_idx in skf.split(X, y):
        model = lgb.LGBMClassifier(
            n_estimators=n_rounds, learning_rate=0.08, num_leaves=63,
            subsample=0.8, colsample_bytree=0.8, random_state=seed, verbose=-1,
        )
        model.fit(X.iloc[tr_idx], y[tr_idx],
                  categorical_feature=cat_cols if cat_cols else "auto")
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    return roc_auc_score(y, oof)

base_cols = NUM_FEATS + CAT_FEATS
baseline_auc = quick_cv_auc(train_fe, base_cols, CAT_FEATS)
print(f"baseline (raw features only):        AUC = {baseline_auc:.5f}")

rng = np.random.default_rng(0)
train_fe["_noise"] = rng.random(len(train_fe))
noise_auc = quick_cv_auc(train_fe, base_cols + ["_noise"], CAT_FEATS)
noise_bar = abs(noise_auc - baseline_auc)
print(f"baseline + random noise column:      AUC = {noise_auc:.5f}   (bar = {noise_bar:.5f})")
train_fe.drop(columns=["_noise"], inplace=True)

engineered_cols = base_cols + ENGINEERED
engineered_auc = quick_cv_auc(train_fe, engineered_cols, CAT_FEATS)
gain = engineered_auc - baseline_auc
print(f"baseline + engineered features:      AUC = {engineered_auc:.5f}   (gain = {gain:+.5f})")

ablation_log = [
    {"stage": "baseline (raw)", "n_features": len(base_cols), "auc": baseline_auc},
    {"stage": "+ noise column (sanity)", "n_features": len(base_cols) + 1, "auc": noise_auc},
    {"stage": "+ interaction/ratio features", "n_features": len(engineered_cols), "auc": engineered_auc},
]

if gain > noise_bar:
    print(f"\n-> Gain ({gain:.5f}) clears the noise bar ({noise_bar:.5f}). Keeping the engineered features.")
else:
    print(f"\n-> Gain ({gain:.5f}) does NOT clear the noise bar ({noise_bar:.5f}).")
    print("   Keeping them anyway for the interpretability/diversity they add to the ensemble, but")
    print("   this is the honest result: don't oversell this feature set on the leaderboard.")


In [ ]:
DIGIT_SOURCE_COLS = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
                     "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
                     "Environmental_Concern_Level"]

def add_digit_features(df, cols=DIGIT_SOURCE_COLS, lo=-4, hi=3):
    out = df.copy()
    for c in cols:
        for k in range(lo, hi + 1):
            out[f"{c}_digit{k}"] = (out[c].fillna(0) // (10.0 ** k) % 10).astype("int8")
    return out

train_fe = add_digit_features(train_fe)
test_fe = add_digit_features(test_fe)
DIGIT_COLS = [f"{c}_digit{k}" for c in DIGIT_SOURCE_COLS for k in range(-4, 4)]
print(f"Added {len(DIGIT_COLS)} digit features from {len(DIGIT_SOURCE_COLS)} numeric columns")

digit_cols_all = engineered_cols + DIGIT_COLS
digit_auc = quick_cv_auc(train_fe, digit_cols_all, CAT_FEATS)
digit_gain = digit_auc - engineered_auc
print(f"+ digit features:                    AUC = {digit_auc:.5f}   (gain = {digit_gain:+.5f})")
ablation_log.append({"stage": "+ digit features", "n_features": len(digit_cols_all), "auc": digit_auc})

if digit_gain > noise_bar:
    print(f"-> Clears the noise bar ({noise_bar:.5f}). Keeping digit features.")
else:
    print(f"-> Does NOT clear the noise bar ({noise_bar:.5f}) on this quick check -- worth re-checking")
    print("   against the real 5-model CV below before trusting it fully.")

### Digit features — exposing synthetic-generation artifacts

Two independent public solutions on this exact competition (one bronze, one silver) found the same thing:
extracting each **decimal digit** of the raw numeric columns as its own feature gives trees a way to notice
generator-specific patterns (rounding behavior, repeated decimals) that they can't easily reconstruct from
the raw float via a handful of splits. For a value `x`, `digit_k(x) = (x // 10^k) % 10` for digit positions
`k = -4..3` (four fractional, four integer positions).


In [ ]:
FREQ_SOURCE_COLS = base_cols + ENGINEERED + DIGIT_COLS

def add_frequency_features(train_df, test_df, cols):
    tr, te = train_df.copy(), test_df.copy()
    for c in cols:
        freq = pd.concat([tr[c].astype(str), te[c].astype(str)], ignore_index=True).value_counts(normalize=True)
        tr[f"{c}_freq"] = tr[c].astype(str).map(freq).astype("float32")
        te[f"{c}_freq"] = te[c].astype(str).map(freq).astype("float32")
    return tr, te

train_fe, test_fe = add_frequency_features(train_fe, test_fe, FREQ_SOURCE_COLS)
FREQ_COLS = [f"{c}_freq" for c in FREQ_SOURCE_COLS]
print(f"Added {len(FREQ_COLS)} frequency features")

freq_cols_all = digit_cols_all + FREQ_COLS
freq_auc = quick_cv_auc(train_fe, freq_cols_all, CAT_FEATS)
freq_gain = freq_auc - digit_auc
print(f"+ frequency features:                AUC = {freq_auc:.5f}   (gain = {freq_gain:+.5f})")
ablation_log.append({"stage": "+ frequency features", "n_features": len(freq_cols_all), "auc": freq_auc})

if freq_gain > noise_bar:
    print(f"-> Clears the noise bar ({noise_bar:.5f}). Keeping frequency features.")
else:
    print(f"-> Does NOT clear the noise bar ({noise_bar:.5f}) on this quick check -- worth re-checking")
    print("   against the real 5-model CV below before trusting it fully.")


### Frequency encoding — how common is this exact value?

For every column built so far (raw + engineered + digit), replace each value with its normalized frequency
across **train + test combined**. This uses no label information (so it's leakage-free by construction) and
gives the model a second, distributional view of the same information: is this a common value, or a rare
one? Synthetic generators often leave rare-value patterns that this exposes directly.


In [ ]:
FREQ_SOURCE_COLS = base_cols + ENGINEERED + DIGIT_COLS

def add_frequency_features(train_df, test_df, cols):
    tr, te = train_df.copy(), test_df.copy()
    for c in cols:
        freq = pd.concat([tr[c].astype(str), te[c].astype(str)], ignore_index=True).value_counts(normalize=True)
        tr[f"{c}_freq"] = tr[c].astype(str).map(freq).astype("float32")
        te[f"{c}_freq"] = te[c].astype(str).map(freq).astype("float32")
    return tr, te

train_fe, test_fe = add_frequency_features(train_fe, test_fe, FREQ_SOURCE_COLS)
FREQ_COLS = [f"{c}_freq" for c in FREQ_SOURCE_COLS]
print(f"Added {len(FREQ_COLS)} frequency features")

freq_cols_all = digit_cols_all + FREQ_COLS
freq_auc = quick_cv_auc(train_fe, freq_cols_all, CAT_FEATS)
freq_gain = freq_auc - digit_auc
print(f"+ frequency features:                AUC = {freq_auc:.5f}   (gain = {freq_gain:+.5f})")
ablation_log.append({"stage": "+ frequency features", "n_features": len(freq_cols_all), "auc": freq_auc})

if freq_gain > noise_bar:
    print(f"-> Clears the noise bar ({noise_bar:.5f}). Keeping frequency features.")
else:
    print(f"-> Does NOT clear the noise bar ({noise_bar:.5f}) on this quick check -- worth re-checking")
    print("   against the real 5-model CV below before trusting it fully.")


### Dropping constants and exact duplicates

Cheap cleanup before the real training: any column that never varies, or that's a perfect (correlation = 1.0)
copy of another column, adds nothing and just slows every model down.

In [ ]:
candidate_cols = NUM_FEATS + ENGINEERED + DIGIT_COLS + FREQ_COLS  # numeric-only; CAT_FEATS always kept

const_cols = [c for c in candidate_cols if train_fe[c].nunique() <= 1]

corr = train_fe[[c for c in candidate_cols if c not in const_cols]].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
dup_cols = [c for c in upper.columns if (upper[c] == 1.0).any()]

DROP_COLS = sorted(set(const_cols) | set(dup_cols))
print(f"Dropping {len(const_cols)} constant + {len(dup_cols)} perfectly-duplicate columns: {DROP_COLS}")

FINAL_NUMERIC_ENGINEERED = [c for c in (ENGINEERED + DIGIT_COLS + FREQ_COLS) if c not in DROP_COLS]
del corr, upper


### Feature engineering summary


In [ ]:
ablation_df = pd.DataFrame(ablation_log)
ablation_df["gain_vs_prev"] = ablation_df["auc"].diff()
ablation_df

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(range(len(ablation_df)), ablation_df["auc"], marker="o", color=PALETTE[0], lw=2)
for i, row in ablation_df.iterrows():
    ax.annotate(f"{row['auc']:.4f}", (i, row["auc"]), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8.5)
ax.set_xticks(range(len(ablation_df)))
ax.set_xticklabels(ablation_df["stage"], rotation=20, ha="right", fontsize=8.5)
ax.set_ylabel("quick-check OOF AUC")
ax.set_title("Feature engineering ablation (fast LightGBM check, not the final model)")
plt.tight_layout(); plt.show()

ablation_df


## 4. Modeling

Final feature set = raw features + engineered/digit/frequency features (post-dedup). One fixed 5-fold
stratified split is reused for **every** model below, so their OOF predictions are directly comparable and
safely blendable.


In [ ]:
FINAL_FEATURES = NUM_FEATS + CAT_FEATS + FINAL_NUMERIC_ENGINEERED
print(f"{len(FINAL_FEATURES)} final features")

y = (train_fe[TARGET] == "Yes").astype(int).values
X = train_fe[FINAL_FEATURES].copy()
X_test = test_fe[FINAL_FEATURES].copy()

for c in CAT_FEATS:
    X[c] = X[c].astype("category")
    X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(f"{N_FOLDS}-fold StratifiedKFold(shuffle=True, random_state={SEED})")


### Hyperparameter tuning (Optuna)

A quick Optuna search for LightGBM, evaluated with a **smaller, faster** CV (3 folds, capped rounds, and
optionally a row subsample) than the real training below — that's a legitimate speed/quality trade-off for a
*search*, since we only need relative comparisons between hyperparameter sets, not a final leaderboard-grade
number. The winning params are then used for the real 5-fold, full-data training that follows.

Set `TUNE_LGB = False` to skip straight to sensible hand-picked defaults (useful while you're iterating on
everything else and don't want to spend the tuning budget every run).


In [ ]:
TUNE_LGB = True
N_TRIALS = 30
TUNE_SUBSAMPLE = 150_000   # rows used for the search only, not for final training

def lgb_objective(trial):
    params = dict(
        n_estimators=1500,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 31, 255),
        max_depth=trial.suggest_int("max_depth", 4, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        random_state=SEED, n_jobs=-1, verbose=-1,
    )

    idx = np.random.default_rng(SEED).choice(len(X), min(TUNE_SUBSAMPLE, len(X)), replace=False)
    Xs, ys = X.iloc[idx].reset_index(drop=True), y[idx]
    skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    for tr_idx, va_idx in skf3.split(Xs, ys):
        model = lgb.LGBMClassifier(**params)
        model.fit(Xs.iloc[tr_idx], ys[tr_idx],
                  eval_set=[(Xs.iloc[va_idx], ys[va_idx])], eval_metric="auc",
                  categorical_feature=CAT_FEATS,
                  callbacks=[lgb.early_stopping(80, verbose=False)])
        scores.append(roc_auc_score(ys[va_idx], model.predict_proba(Xs.iloc[va_idx])[:, 1]))
    return float(np.mean(scores))

if TUNE_LGB:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=False)
    print(f"Best CV AUC during search: {study.best_value:.5f}")
    print("Best params:", study.best_params)
    TUNED_LGB_PARAMS = dict(study.best_params)
else:
    TUNED_LGB_PARAMS = dict(learning_rate=0.03, num_leaves=127, max_depth=-1,
                            min_child_samples=40, subsample=0.8, colsample_bytree=0.7,
                            reg_alpha=0.1, reg_lambda=0.5)
    print("Skipping the search, using hand-picked defaults:", TUNED_LGB_PARAMS)


Optuna's `plot_optimization_history` and `plot_param_importances` are worth showing here too — they're
genuinely interesting for readers (which params actually mattered) and Optuna renders them as Plotly figures,
which look sharp in a Kaggle notebook.

In [ ]:
if TUNE_LGB:
    from optuna.visualization import plot_optimization_history, plot_param_importances
    plot_optimization_history(study).show()
    plot_param_importances(study).show()


### Fold-safe target encoding

Target encoding (replacing a category with the mean target for that category) is powerful but leaks badly
if done naively — a row's own label ends up baked into the value used to predict it. The safe version, used
here for the six categorical columns:

1. **Within each outer training fold**, further split into 5 inner folds and encode each inner-validation row
   using only the mean computed from the *other* inner folds (so every training row's encoding excludes its
   own label).
2. For the outer validation fold and the test set, encode using the mean computed from **all** of the outer
   fold's training data (those rows' labels never touch validation/test, so this is safe).
3. Smooth every mean toward the global rate — categories with few examples get pulled toward the overall
   average instead of reporting a noisy small-sample mean.

This is computed **once per outer fold** and shared across all four models below, rather than recomputed
per model.


In [ ]:
def target_encode_fold(train_cat, y_train, valid_cat, test_cat, n_inner_folds=5, smoothing=20, seed=SEED):
    global_mean = y_train.mean()
    inner_skf = StratifiedKFold(n_splits=n_inner_folds, shuffle=True, random_state=seed)

    train_enc = pd.DataFrame(index=train_cat.index)
    valid_enc = pd.DataFrame(index=valid_cat.index)
    test_enc = pd.DataFrame(index=test_cat.index)

    # .map() on a categorical Series (or even on a plain Series built from .values of one)
    # silently returns categorical output -- decategorize first so results come back as float.
    train_plain = train_cat.astype(object)
    valid_plain = valid_cat.astype(object)
    test_plain = test_cat.astype(object)

    for c in train_cat.columns:
        oof_col = np.full(len(train_cat), np.nan)
        for in_idx, out_idx in inner_skf.split(train_plain, y_train):
            stats = pd.Series(y_train[in_idx]).groupby(train_plain[c].values[in_idx]).agg(["mean", "count"])
            smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
            oof_col[out_idx] = pd.Series(train_plain[c].values[out_idx]).map(smoothed).fillna(global_mean).values
        train_enc[f"{c}_TE"] = oof_col.astype("float32")

        full_stats = pd.Series(y_train).groupby(train_plain[c].values).agg(["mean", "count"])
        full_smoothed = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (full_stats["count"] + smoothing)
        valid_enc[f"{c}_TE"] = valid_plain[c].map(full_smoothed).fillna(global_mean).astype("float32").values
        test_enc[f"{c}_TE"] = test_plain[c].map(full_smoothed).fillna(global_mean).astype("float32").values

    return train_enc.reset_index(drop=True), valid_enc.reset_index(drop=True), test_enc.reset_index(drop=True)

TE_COLS = [f"{c}_TE" for c in CAT_FEATS]
TE_CACHE = []
for tr_idx, va_idx in folds:
    tr_enc, va_enc, te_enc = target_encode_fold(
        X.iloc[tr_idx][CAT_FEATS].reset_index(drop=True), y[tr_idx],
        X.iloc[va_idx][CAT_FEATS].reset_index(drop=True), X_test[CAT_FEATS].reset_index(drop=True),
    )
    TE_CACHE.append({"train": tr_enc, "valid": va_enc, "test": te_enc})

print(f"Target-encoded {len(CAT_FEATS)} categorical columns, cached for all {N_FOLDS} folds.")


### Model 1 — LightGBM

In [ ]:
lgb_oof = np.zeros(len(X))
lgb_test = np.zeros(len(X_test))
lgb_models = []

lgb_params = dict(
    n_estimators=6000,
    subsample_freq=1,
    random_state=SEED, n_jobs=-1, verbose=-1,
    **TUNED_LGB_PARAMS,
)
if USE_GPU:
    # LightGBM's GPU histogram builder packs bin indices into 8 bits, so max_bin must be <= 255
    # on GPU (511 only works on CPU).
    lgb_params["device_type"] = "gpu"
    lgb_params["max_bin"] = 255
else:
    lgb_params["max_bin"] = 511

for fold, (tr_idx, va_idx) in enumerate(folds):
    X_tr = pd.concat([X.iloc[tr_idx].reset_index(drop=True), TE_CACHE[fold]["train"]], axis=1)
    X_va = pd.concat([X.iloc[va_idx].reset_index(drop=True), TE_CACHE[fold]["valid"]], axis=1)
    X_te = pd.concat([X_test.reset_index(drop=True), TE_CACHE[fold]["test"]], axis=1)

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y[tr_idx],
        eval_set=[(X_va, y[va_idx])],
        eval_metric="auc",
        categorical_feature=CAT_FEATS,
        callbacks=[lgb.early_stopping(200, verbose=False)],
    )
    lgb_oof[va_idx] = model.predict_proba(X_va)[:, 1]
    lgb_test += model.predict_proba(X_te)[:, 1] / N_FOLDS
    lgb_models.append(model)
    fold_auc = roc_auc_score(y[va_idx], lgb_oof[va_idx])
    print(f"  fold {fold}: AUC = {fold_auc:.5f}  (best_iteration={model.best_iteration_})")

lgb_auc = roc_auc_score(y, lgb_oof)
print(f"\nLightGBM OOF AUC: {lgb_auc:.5f}")

### Model 2 — XGBoost

In [ ]:
X_xgb = X.copy()
X_test_xgb = X_test.copy()

xgb_oof = np.zeros(len(X))
xgb_test = np.zeros(len(X_test))

xgb_params = dict(
    n_estimators=6000, learning_rate=0.02, max_depth=7,
    min_child_weight=10, subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0, max_bin=1024,
    tree_method="hist", enable_categorical=True,
    eval_metric="auc", random_state=SEED, n_jobs=-1,
)
if USE_GPU:
    xgb_params["device"] = "cuda"

for fold, (tr_idx, va_idx) in enumerate(folds):
    X_tr = pd.concat([X_xgb.iloc[tr_idx].reset_index(drop=True), TE_CACHE[fold]["train"]], axis=1)
    X_va = pd.concat([X_xgb.iloc[va_idx].reset_index(drop=True), TE_CACHE[fold]["valid"]], axis=1)
    X_te = pd.concat([X_test_xgb.reset_index(drop=True), TE_CACHE[fold]["test"]], axis=1)

    model = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=250)
    model.fit(
        X_tr, y[tr_idx],
        eval_set=[(X_va, y[va_idx])],
        verbose=False,
    )
    xgb_oof[va_idx] = model.predict_proba(X_va)[:, 1]
    xgb_test += model.predict_proba(X_te)[:, 1] / N_FOLDS
    fold_auc = roc_auc_score(y[va_idx], xgb_oof[va_idx])
    print(f"  fold {fold}: AUC = {fold_auc:.5f}  (best_iteration={model.best_iteration})")

xgb_auc = roc_auc_score(y, xgb_oof)
print(f"\nXGBoost OOF AUC: {xgb_auc:.5f}")


### Model 3 — CatBoost

In [ ]:
cat_idx = [X.columns.get_loc(c) for c in CAT_FEATS]

cb_oof = np.zeros(len(X))
cb_test = np.zeros(len(X_test))

X_cb = X.copy()
X_test_cb = X_test.copy()
for c in CAT_FEATS:
    X_cb[c] = X_cb[c].astype(str)
    X_test_cb[c] = X_test_cb[c].astype(str)

cb_params = dict(
    iterations=6000, learning_rate=0.02, depth=8,
    l2_leaf_reg=5, loss_function="Logloss", eval_metric="AUC",
    border_count=254, random_seed=SEED, verbose=False,
)
if USE_GPU:
    cb_params["task_type"] = "GPU"

for fold, (tr_idx, va_idx) in enumerate(folds):
    X_tr = pd.concat([X_cb.iloc[tr_idx].reset_index(drop=True), TE_CACHE[fold]["train"]], axis=1)
    X_va = pd.concat([X_cb.iloc[va_idx].reset_index(drop=True), TE_CACHE[fold]["valid"]], axis=1)
    X_te = pd.concat([X_test_cb.reset_index(drop=True), TE_CACHE[fold]["test"]], axis=1)

    train_pool = Pool(X_tr, y[tr_idx], cat_features=cat_idx)
    valid_pool = Pool(X_va, y[va_idx], cat_features=cat_idx)
    test_pool = Pool(X_te, cat_features=cat_idx)

    model = CatBoostClassifier(**cb_params)
    model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=250, use_best_model=True)

    cb_oof[va_idx] = model.predict_proba(valid_pool)[:, 1]
    cb_test += model.predict_proba(test_pool)[:, 1] / N_FOLDS
    fold_auc = roc_auc_score(y[va_idx], cb_oof[va_idx])
    print(f"  fold {fold}: AUC = {fold_auc:.5f}  (best_iteration={model.get_best_iteration()})")

cb_auc = roc_auc_score(y, cb_oof)
print(f"\nCatBoost OOF AUC: {cb_auc:.5f}")


### Model 4 — Logistic Regression (linear baseline, for ensemble diversity)

The gradient boosters above capture non-linear interactions well but tend to correlate with each other.
A properly cross-validated linear model — scaling and one-hot encoding **fit inside each fold**, not on the
whole training set — adds a genuinely different error pattern to blend against, which is usually worth more
to an ensemble than a fourth tree model.


In [ ]:
lr_oof = np.zeros(len(X))
lr_test = np.zeros(len(X_test))

preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATS + [c for c in ENGINEERED if c != "Range_Anxiety_Ordinal"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
])

X_lr = train_fe[FINAL_FEATURES].copy()
X_test_lr = test_fe[FINAL_FEATURES].copy()

for fold, (tr_idx, va_idx) in enumerate(folds):
    pipe = Pipeline([
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced")),
    ])
    pipe.fit(X_lr.iloc[tr_idx], y[tr_idx])
    lr_oof[va_idx] = pipe.predict_proba(X_lr.iloc[va_idx])[:, 1]
    lr_test += pipe.predict_proba(X_test_lr)[:, 1] / N_FOLDS
    fold_auc = roc_auc_score(y[va_idx], lr_oof[va_idx])
    print(f"  fold {fold}: AUC = {fold_auc:.5f}")

lr_auc = roc_auc_score(y, lr_oof)
print(f"\nLogistic Regression OOF AUC: {lr_auc:.5f}")


## 5. Comparing models and building the blend

In [ ]:
results = pd.DataFrame({
    "model": ["LightGBM", "XGBoost", "CatBoost", "Logistic Regression"],
    "OOF AUC": [lgb_auc, xgb_auc, cb_auc, lr_auc],
}).sort_values("OOF AUC", ascending=False)
results


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(results["model"], results["OOF AUC"], color=PALETTE[:len(results)])
ax.set_xlim(results["OOF AUC"].min() - 0.01, results["OOF AUC"].max() + 0.005)
ax.set_xlabel("OOF ROC-AUC")
ax.set_title("Model comparison (out-of-fold)")
plt.tight_layout(); plt.show()


We now find blend weights that maximize OOF AUC. Rather than a plain average, we search over the
simplex of weights (all non-negative, summing to 1) with a simple grid/optimizer — this is still fit purely
on OOF predictions, so it costs us nothing in terms of leakage: the labels being optimized against are exactly
the ones each model never saw during its own training.

In [ ]:
from scipy.optimize import minimize

oof_matrix = np.column_stack([lgb_oof, xgb_oof, cb_oof, lr_oof])
test_matrix = np.column_stack([lgb_test, xgb_test, cb_test, lr_test])
names = ["LightGBM", "XGBoost", "CatBoost", "LogReg"]

def neg_auc(w):
    w = np.abs(w)
    w = w / w.sum()
    return -roc_auc_score(y, oof_matrix @ w)

res = minimize(neg_auc, x0=np.ones(oof_matrix.shape[1]) / oof_matrix.shape[1], method="Nelder-Mead")
weights = np.abs(res.x)
weights = weights / weights.sum()

for n, w in zip(names, weights):
    print(f"  {n:22s} weight = {w:.3f}")

blend_oof = oof_matrix @ weights
blend_test = test_matrix @ weights
blend_auc = roc_auc_score(y, blend_oof)

print(f"\nSimple average AUC : {roc_auc_score(y, oof_matrix.mean(axis=1)):.5f}")
print(f"Best single model  : {results['OOF AUC'].max():.5f} ({results.iloc[0]['model']})")
print(f"Optimized blend    : {blend_auc:.5f}")


### Stacking — a meta-model instead of a fixed blend

A weighted average is a *linear* combination with weights chosen once. A stacking meta-model can do a bit
better because it can weight the four models differently in different regions of the feature space (e.g.
trust CatBoost more for one category, XGBoost more for another). We fit a simple Logistic Regression on the
four OOF probability columns — this is safe from leakage because every value in `oof_matrix` was produced by
a model that never saw that row during its own training, so the meta-model is also effectively cross-validated.


In [ ]:
from sklearn.model_selection import cross_val_predict

meta = LogisticRegression(max_iter=1000)
# cross_val_predict here re-validates the meta-model itself on held-out folds of the OOF matrix,
# so we're not just fitting-and-scoring on the same rows.
stack_oof = cross_val_predict(meta, oof_matrix, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                              method="predict_proba")[:, 1]
meta.fit(oof_matrix, y)
stack_test = meta.predict_proba(test_matrix)[:, 1]
stack_auc = roc_auc_score(y, stack_oof)

print(f"Optimized blend AUC     : {blend_auc:.5f}")
print(f"Stacking meta-model AUC : {stack_auc:.5f}")
print("meta-model coefficients:", dict(zip(names, meta.coef_[0].round(3))))

if stack_auc > blend_auc:
    print("\n-> Stacking wins. Using it for the final submission.")
    FINAL_oof, FINAL_test, FINAL_auc, FINAL_label = stack_oof, stack_test, stack_auc, "stacking"
else:
    print("\n-> The blend wins (or ties). Using it for the final submission.")
    FINAL_oof, FINAL_test, FINAL_auc, FINAL_label = blend_oof, blend_test, blend_auc, "optimized blend"


### ROC and Precision-Recall curves (OOF, final model vs. best single model)

The ROC curve is the metric we're scored on, made visual. The PR curve is the one to also check on an
imbalanced target like this one — it's far more sensitive to how well the model ranks the minority class,
which ROC-AUC can flatter.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

best_single_name = results.iloc[0]["model"]
best_single_oof = {"LightGBM": lgb_oof, "XGBoost": xgb_oof, "CatBoost": cb_oof,
                    "Logistic Regression": lr_oof}[best_single_name]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for oof, label, color in [(best_single_oof, f"{best_single_name} (best single)", PALETTE[4]),
                          (FINAL_oof, f"Final ({FINAL_label})", PALETTE[0])]:
    fpr, tpr, _ = roc_curve(y, oof)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{label} (AUC={roc_auc_score(y, oof):.4f})")
axes[0].plot([0, 1], [0, 1], color="0.7", lw=1, ls="--")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curve"); axes[0].legend(fontsize=9, loc="lower right")

for oof, label, color in [(best_single_oof, f"{best_single_name} (best single)", PALETTE[4]),
                          (FINAL_oof, f"Final ({FINAL_label})", PALETTE[0])]:
    prec, rec, _ = precision_recall_curve(y, oof)
    axes[1].plot(rec, prec, color=color, lw=2, label=f"{label} (AP={average_precision_score(y, oof):.4f})")
axes[1].axhline(y.mean(), color="0.7", lw=1, ls="--", label=f"no-skill baseline ({y.mean():.3f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curve"); axes[1].legend(fontsize=9, loc="upper right")

plt.tight_layout(); plt.show()


### Calibration — are the final probabilities trustworthy, not just well-ranked?

In [ ]:
from sklearn.calibration import calibration_curve

frac_pos, mean_pred = calibration_curve(y, FINAL_oof, n_bins=15, strategy="quantile")

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot([0, 1], [0, 1], color="0.7", lw=1, ls="--", label="perfectly calibrated")
ax.plot(mean_pred, frac_pos, marker="o", color=PALETTE[0], lw=2, label=f"final ({FINAL_label}, OOF)")
ax.set_xlabel("mean predicted probability (per bin)")
ax.set_ylabel("observed fraction positive (per bin)")
ax.set_title("Calibration curve — matters for interpreting probabilities,\nnot for the AUC score itself")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print("ROC-AUC only cares about ranking, so calibration doesn't change your leaderboard score — but a")
print("well-calibrated model is more trustworthy if you ever use these probabilities for anything besides")
print("submission (e.g. picking a threshold, or explaining a prediction to someone).")


### Where is the model actually weak? A per-subgroup breakdown

Overall OOF AUC hides a lot. Breaking it down by each categorical value shows whether the final model is
uniformly good, or quietly bad on one segment (e.g. a specific `City_Type` or `Current_Car_Type`) that a
single headline number would never surface. This also doubles as an error-analysis starting point: a
segment that consistently underperforms is a candidate for its own interaction feature.

In [ ]:
subgroup_rows = []
for c in CAT_FEATS:
    for level in train_fe[c].dropna().unique():
        mask = (train_fe[c] == level).values
        if mask.sum() < 200 or y[mask].sum() == 0 or y[mask].sum() == mask.sum():
            continue  # too few rows, or only one class present -- AUC isn't defined
        subgroup_rows.append({
            "column": c, "level": str(level),
            "auc": roc_auc_score(y[mask], FINAL_oof[mask]),
            "n": int(mask.sum()),
        })

subgroup_df = pd.DataFrame(subgroup_rows).sort_values("auc")
overall_final_auc = roc_auc_score(y, FINAL_oof)

fig, ax = plt.subplots(figsize=(9, max(4, 0.34 * len(subgroup_df))))
colors = [PALETTE[3] if a < overall_final_auc else PALETTE[2] for a in subgroup_df["auc"]]
labels = subgroup_df["column"] + " = " + subgroup_df["level"]
bars = ax.barh(labels, subgroup_df["auc"] - overall_final_auc, left=overall_final_auc, color=colors)
ax.axvline(overall_final_auc, color="black", lw=1, ls="--", label=f"overall ({overall_final_auc:.4f})")
for bar, n in zip(bars, subgroup_df["n"]):
    ax.text(bar.get_x() + bar.get_width(), bar.get_y() + bar.get_height() / 2, f"  n={n:,}",
            va="center", fontsize=7.5, color="0.35")
ax.set_xlabel("subgroup OOF AUC")
ax.set_title("Model performance by categorical subgroup\n(orange = below the overall average)")
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout(); plt.show()

worst = subgroup_df.iloc[0]
print(f"Weakest segment: {worst['column']} = {worst['level']}  (AUC {worst['auc']:.4f}, n={worst['n']:,})")


### Feature importance (LightGBM, averaged across folds)

In [ ]:
imp = pd.DataFrame({
    "feature": FINAL_FEATURES + TE_COLS,
    "importance": np.mean([m.feature_importances_ for m in lgb_models], axis=0),
}).sort_values("importance", ascending=False).head(15).iloc[::-1]

norm = (imp["importance"] - imp["importance"].min()) / (imp["importance"].max() - imp["importance"].min() + 1e-9)
colors = plt.cm.viridis(0.15 + 0.7 * norm)

fig, ax = plt.subplots(figsize=(7.5, 6))
bars = ax.barh(imp["feature"], imp["importance"], color=colors)
for bar, v in zip(bars, imp["importance"]):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f"  {v:,.0f}",
            va="center", fontsize=8.5, color="0.25")
ax.set_title("Top 15 features by LightGBM gain (averaged over folds)", fontsize=12)
ax.set_xlabel("importance (gain)")
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout(); plt.show()


## 6. Submission

In [ ]:
submission = test[[ID_COL]].copy()
submission[TARGET] = FINAL_test
print(f"Submitting the {FINAL_label} (OOF AUC = {FINAL_auc:.5f})")

assert list(submission.columns) == list(sub.columns), "Column mismatch with sample_submission!"
assert len(submission) == len(sub), "Row count mismatch with sample_submission!"
assert submission[TARGET].between(0, 1).all(), "Predictions must be probabilities in [0, 1]!"

submission.to_csv("submission.csv", index=False)
submission.head()
